# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# Imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Load environment
load_dotenv(override=True)

or_api_key      = os.getenv('OR_API_KEY')
groq_api_key    = os.getenv('GROQ_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
gemini_api_key  = os.getenv('GEMINI_API_KEY')

# Clients -- all via the OpenAI-compatible interface (Week 2, Day 1 technique)
or_client       = OpenAI(api_key=or_api_key,       base_url="https://openrouter.ai/api/v1")
groq_client     = OpenAI(api_key=groq_api_key,     base_url="https://api.groq.com/openai/v1")
anthropic_client= OpenAI(api_key=anthropic_api_key, base_url="https://api.anthropic.com/v1")
gemini_client   = OpenAI(api_key=gemini_api_key,   base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
ollama_client   = OpenAI(api_key="ollama",          base_url="http://localhost:11434/v1")

print("All clients initialised!")
for name, key in [("OpenRouter", or_api_key), ("Groq", groq_api_key),
                  ("Anthropic", anthropic_api_key), ("Gemini", gemini_api_key)]:
    print(f"  {name}: {'\u2713 ' + key[:8] if key else '\u2717 not set'}")

In [ ]:
# System prompt -- adds deep technical expertise (Week 2, Day 3 technique)

system_prompt = """You are an expert technical tutor specialising in LLM Engineering and AI development.
Your students are software engineers learning to build AI-powered applications.

When answering:
- Provide clear, structured explanations with concrete code examples in markdown code blocks
- Draw simple ASCII diagrams for complex architectures when helpful
- Explain *why* things work, not just *what* they do
- Relate concepts to real-world LLM/AI engineering use cases
- Suggest related topics the student might want to explore next

You have access to a knowledge-base tool -- use it when a student asks about a recognised topic
(tokenization, embeddings, attention, RAG, fine-tuning, prompting, tools/function-calling).

Always respond in markdown."""

# --- Available models (Week 2, Day 1) ---
# Prioritise no-credit / low-cost options first.
MODELS = {}

# Free/low-cost cloud via Groq (if key is set)
if groq_api_key:
    MODELS["Llama 3.3 70B (Groq -- fast)"] = {"client": "groq", "model": "llama-3.3-70b-versatile"}
    MODELS["GPT-OSS 20B (Groq)"] = {"client": "groq", "model": "openai/gpt-oss-20b"}

# Local no-credit model via Ollama (always included)
MODELS["Llama 3.2 3B (Ollama -- local)"] = {"client": "ollama", "model": "llama3.2:3b"}

# Optional paid/additional providers (only shown when keys exist)
if or_api_key:
    MODELS["GPT-4o-mini (OpenRouter)"] = {"client": "or", "model": "openai/gpt-4o-mini"}
if anthropic_api_key:
    MODELS["Claude Haiku 4.5 (Anthropic)"] = {"client": "anthropic", "model": "claude-haiku-4-5"}
if gemini_api_key:
    MODELS["Gemini 2.0 Flash (Google)"] = {"client": "gemini", "model": "gemini-2.0-flash"}

_client_map = {
    "or": or_client,
    "anthropic": anthropic_client,
    "gemini": gemini_client,
    "groq": groq_client,
    "ollama": ollama_client,
}


def get_client(model_name: str):
    cfg = MODELS[model_name]
    return _client_map[cfg["client"]], cfg["model"]


print("Models available:")
for name in MODELS:
    print(f"  - {name}")

In [ ]:
# Bonus tool: tiny built-in knowledge base

KB = {
    "tokenization": "Tokenization converts text into tokens that the model can process. Modern LLM tokenizers use subword/BPE-style vocabularies to balance efficiency and coverage.",
    "embeddings": "Embeddings map text to dense vectors where semantic similarity is represented by geometric closeness. They power retrieval, clustering, and semantic search.",
    "attention": "Attention lets each token dynamically weight other tokens, enabling context-aware representations and strong sequence modeling.",
    "rag": "RAG (Retrieval-Augmented Generation) retrieves external context at runtime and injects it into prompts to improve factuality and domain coverage.",
    "fine-tuning": "Fine-tuning updates model weights on curated data for style/domain adaptation, while prompting keeps base weights unchanged.",
    "prompting": "Prompting is the art of structuring instructions/context/examples so the model produces better outputs without retraining.",
    "tools": "Tool/function calling lets the model request external functions (APIs, DB lookups, calculators), then continue reasoning with returned results.",
}


def knowledge_base_lookup(topic: str) -> str:
    query = (topic or "").strip().lower()
    if not query:
        return "Please provide a topic to look up."

    # fuzzy-ish match so users can type natural variants
    for key, value in KB.items():
        if query in key or key in query:
            return f"Topic: {key}\n\n{value}"

    topics = ", ".join(sorted(KB.keys()))
    return f"I don't have that topic yet. Try one of: {topics}"


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "knowledge_base_lookup",
            "description": "Look up a short explanation for a core LLM engineering concept.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The concept to retrieve (e.g., tokenization, embeddings, rag, prompting)",
                    }
                },
                "required": ["topic"],
            },
        },
    }
]

print("Knowledge-base tool ready!")

In [ ]:
# Core chat logic with tool-calling + streaming
# Day 4 pattern: keep looping while the model asks for tools.


def _serialise_tool_call(tc):
    if hasattr(tc, "model_dump"):
        return tc.model_dump()
    return {
        "id": tc.id,
        "type": "function",
        "function": {
            "name": tc.function.name,
            "arguments": tc.function.arguments,
        },
    }


def _history_to_messages(history):
    messages = []
    for turn in history or []:
        if isinstance(turn, dict) and "role" in turn and "content" in turn:
            messages.append({"role": turn["role"], "content": turn["content"]})
            continue
        if isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_msg, assistant_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})
    return messages


def _run_tool_loop(client, model_id: str, messages: list, max_rounds: int = 4):
    """Resolve one or more tool calls before final answer stream."""
    working = list(messages)

    for _ in range(max_rounds):
        response = client.chat.completions.create(
            model=model_id,
            messages=working,
            tools=TOOLS,
            tool_choice="auto",
            stream=False,
            temperature=0.3,
        )

        choice = response.choices[0]
        msg = choice.message
        finish_reason = getattr(choice, "finish_reason", None)
        tool_calls = getattr(msg, "tool_calls", None) or []

        # No tool request -> we are ready to do final streamed generation
        if finish_reason != "tool_calls" or not tool_calls:
            if msg.content:
                working.append({"role": "assistant", "content": msg.content})
            return working

        working.append(
            {
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [_serialise_tool_call(tc) for tc in tool_calls],
            }
        )

        for tc in tool_calls:
            fn_name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")

            if fn_name == "knowledge_base_lookup":
                result = knowledge_base_lookup(args.get("topic", ""))
            else:
                result = f"Unknown tool: {fn_name}"

            working.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "name": fn_name,
                    "content": result,
                }
            )

    # Safety fallback if a model keeps calling tools repeatedly
    working.append(
        {
            "role": "assistant",
            "content": "I reached the tool-call limit for this turn. Please refine your question and try again.",
        }
    )
    return working


def chat_stream(message, history, model_name):
    try:
        client, model_id = get_client(model_name)

        messages = [{"role": "system", "content": system_prompt}]
        messages.extend(_history_to_messages(history))
        messages.append({"role": "user", "content": message})

        prepared_messages = _run_tool_loop(client, model_id, messages)

        stream = client.chat.completions.create(
            model=model_id,
            messages=prepared_messages,
            stream=True,
            temperature=0.3,
        )

        partial = ""
        for chunk in stream:
            delta = chunk.choices[0].delta
            token = getattr(delta, "content", None) or ""
            partial += token
            yield partial

    except Exception as e:
        yield f"Error: {e}"


print("Streaming + tool loop ready!")

In [ ]:
# Gradio app (Day 2) + model switcher (Day 1) + streaming (Day 5)

model_dropdown = gr.Dropdown(
    choices=list(MODELS.keys()),
    value=list(MODELS.keys())[0],
    label="Choose model/provider",
)

examples = [
    ["Explain retrieval augmented generation (RAG) with an architecture sketch."],
    ["When should I use fine-tuning instead of prompt engineering?"],
    ["Use the knowledge base tool to teach me tokenization."],
    ["Give me a practical checklist for production LLM observability."],
]

ui = gr.ChatInterface(
    fn=chat_stream,
    type="messages",
    additional_inputs=[model_dropdown],
    title="Technical Q/A Tutor",
    description="Applies Day1-Day5 concepts: multi-provider routing, structured system prompting, tool-calling loop, and streaming answers.",
    examples=examples,
)

ui.queue().launch()

In [ ]:
# Optional Day 5 bonus: local/offline text-to-speech (no API credits)

import tempfile


def text_to_speech_file(text: str):
    """Generate local speech audio if pyttsx3 is available."""
    try:
        import pyttsx3
    except Exception:
        return None, "pyttsx3 not installed. Run: uv add pyttsx3"

    try:
        engine = pyttsx3.init()
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        path = tmp.name
        tmp.close()

        engine.save_to_file((text or "")[:2000], path)
        engine.runAndWait()
        return path, "Audio created locally (offline)."
    except Exception as e:
        return None, f"Local audio generation failed: {e}"


print("Optional local audio helper ready. Example:")
print("audio_path, status = text_to_speech_file('Hello from Week 2 Day 5')")

In [ ]:
audio_path, status = text_to_speech_file('Hello from Week 2 Day 5')

In [ ]:
audio_path, status = text_to_speech_file('Hello, My name is Ed, nice to meet you')